# Jarvis RVC — treino headless (Applio CLI)
Dataset: kuchiriel/jarvis (7 clips, 2.8min, 48kHz). Receita: batch 4, RMVPE, 300 epochs, save/25, pretrained ON.

In [ ]:
import os, subprocess, sys
BASE = "/tmp/program_ml"
os.chdir("/kaggle/working")
def run(cmd, **kw):
    print("+", cmd if isinstance(cmd, str) else " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, shell=isinstance(cmd, str), **kw)
ds = None
for root, dirs, files in os.walk("/kaggle/input"):
    if any(f.endswith(".wav") for f in files):
        ds = root; break
print("DATASET:", ds)
assert ds, "dataset wav nao encontrado em /kaggle/input"
run("nvidia-smi --query-gpu=name,memory.total --format=csv")
import subprocess as _sp
_gpu = _sp.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout
print("GPU:", _gpu.strip())
assert "T4" in _gpu, "GPU errada (%s) — abortando p/ nao gastar quota" % _gpu.strip()


In [ ]:
# install Applio (lean, sem UI/filebrowser)
run("git clone --depth 1 https://github.com/IAHispano/Applio.git " + BASE)
run("apt-get update -qq && apt-get install -y -qq portaudio19-dev ffmpeg")
run([sys.executable, "-m", "pip", "install", "-q", "uv"])
run(["uv", "pip", "install", "-q", "-r", BASE + "/requirements.txt",
     "--extra-index-url", "https://download.pytorch.org/whl/cu128",
     "--index-strategy", "unsafe-best-match", "--system"])
run([sys.executable, "core.py", "prerequisites", "--models", "--exe", "--pretraineds-hifigan"], cwd=BASE)
print("INSTALL OK")

In [ ]:
# preprocess 48k
run([sys.executable, "core.py", "preprocess", "--model-name", "jarvis-rvc",
     "--dataset-path", ds, "--sample-rate", "48000"], cwd=BASE)
print("PREPROCESS OK")

In [ ]:
# extract RMVPE + contentvec, T4x2
run([sys.executable, "core.py", "extract", "--model-name", "jarvis-rvc",
     "--sample-rate", "48000", "--f0-method", "rmvpe",
     "--embedder-model", "contentvec", "--gpu", "0-1"], cwd=BASE)
print("EXTRACT OK")

In [ ]:
# train: batch 4, 300 epochs, save/25, pretrained ON
run([sys.executable, "core.py", "train", "--model-name", "jarvis-rvc",
     "--sample-rate", "48000", "--batch-size", "4",
     "--total-epoch", "300", "--save-every-epoch", "25",
     "--gpu", "0-1", "--pretrained",
     "--save-every-weights"], cwd=BASE)
print("TRAIN OK")

In [ ]:
# index + exporta melhor pth + index p/ /kaggle/working
run([sys.executable, "core.py", "index", "--model-name", "jarvis-rvc",
     "--index-algorithm", "Faiss"], cwd=BASE)
import glob, shutil
exp = BASE + "/logs/jarvis-rvc"
pths = sorted(glob.glob(exp + "/G_*.pth"))
idxs = glob.glob(exp + "/added_*.index") + glob.glob(exp + "/*.index")
print("PTHS:", [p.split("/")[-1] for p in pths[-5:]])
print("INDEX:", [p.split("/")[-1] for p in idxs])
assert pths and idxs, "treino nao gerou pth/index"
shutil.copy(pths[-1], "/kaggle/working/jarvis-best.pth")
shutil.copy(idxs[0], "/kaggle/working/jarvis-best.index")
run([sys.executable, "core.py", "model_information",
     "--pth-path", "/kaggle/working/jarvis-best.pth"], cwd=BASE)
print("EXPORT OK")